In [0]:
# DLT works with three types of Datasets
# Streaming Tables (Permanent/Temporary) a Used as Append Data Sources, Incremental data
# Materialized Views - Used for transformations, aggregations or computations
# Views - Used for intermediate Tranformations, not stored in Target Schema

import dlt

In [0]:
_order_status = spark.conf.get("custom.orderStatus", "NA")  

In [0]:
@dlt.table(name="orders_union_bronze")
def order_delta_append_A():
    return spark.readStream.table("LIVE.orders_bronze")


In [0]:
# Create a streaming table for Orders
@dlt.table(
  table_properties = {"quality": "bronze"},
  comment = "Order bronze table"
)
def orders_bronze():
  df = spark.readStream.table("0624_integ.0624_schema.orders_raw_bronze") 
  return df


In [0]:
# Create a streaming table for Orders Autoloader
# @dlt.table(
#   table_properties = {"quality": "bronze"},
#   comment = "Order Autoloader",
#   name = "orders_autoloader_bronze"
# )
# def func():
#   df = (
#       spark
#       .readStream("cloudFiles")
#       .option("cloudFiles.schemaHints", "o_orderkey long, o_custkey long, o_orderstatus string, o_totalprice decimal(18,2), o_orderdate date, o_orderpriority string, o_clerk string, o_shippriority integer, o_comment string")
#       .option("cloudFiles.schemaLocation","/Volumes/0624_integ/0624_schema/landing/autoloader/schemas/1/")
#       .option("cloudFiles.format", "CSV")
#       .option("pathGlobfilter", "*.csv")
#       .option("cloudFiles.schemaEvolutionMode", "none")
#       .load("/Volumes/0624_integ/0624_schema/landing/files/"  )
# )
#   return df

### 🔹 Corrected Code Here is your corrected `@dlt.table` function:  
@dlt.table(
table_properties={"quality": "bronze"},
comment="Order Autoloader",
name="orders_autoloader_bronze"
)
def func():
  df = (
      spark.readStream
      .format("cloudFiles")  # <-- correct
      .option("cloudFiles.schemaHints", """
          o_orderkey long,
          o_custkey long,
          o_orderstatus string,
          o_totalprice decimal(18,2),
          o_orderdate date,
          o_orderpriority string,
          o_clerk string,
          o_shippriority integer,
          o_comment string
      """)
      .option("cloudFiles.schemaLocation", "/Volumes/0624_integ/0624_schema/landing/autoloader/schemas/1/")
      .option("cloudFiles.format", "csv")
      .option("pathGlobfilter", "*.csv")
      .option("cloudFiles.schemaEvolutionMode", "none")
      .load("/Volumes/0624_integ/0624_schema/landing/files/")
  )
  return df

In [0]:
# Append Flow
@dlt.append_flow(target="orders_union_bronze")
def order_delta_append():
    df = spark.readStream.table("LIVE.orders_bronze")
    return df

# Append Flow
@dlt.append_flow(target="orders_union_bronze")
def order_autoloader_append():
    df = spark.readStream.table("LIVE.orders_autoloader_bronze")
    return df

In [0]:
# Create a Materialized View for Customers
@dlt.table(
  table_properties = {"quality": "bronze"},
  comment = "Customer bronze table",
  name = "customer_bronze"
)
def cust_bronze():
  df = spark.read.table("0624_integ.0624_schema.customer_raw_bronze")
  return df


In [0]:
# Create a View to join orders with customers
@dlt.view(
    comment = "Joined View"
)
def joined_vw():
    df_c = spark.read.table("LIVE.customer_bronze")
    df_o = spark.read.table("LIVE.orders_union_bronze")
    df_join = df_o.join(df_c, how = "left_outer", on=df_c.c_custkey==df_o.o_custkey)
    return df_join  

In [0]:
# Create MV to add new column
from pyspark.sql.functions import current_timestamp, count, sum
@dlt.table(
    table_properties = {"quality": "silver"},
    comment = "Joined table",
    name = "joined_silver"
)
def joined_silver():
    df = spark.read.table("LIVE.joined_vw").withColumn("_insert_date", current_timestamp())
    return df

In [0]:
# Aggregate based on c_nktsegment and find the count of order (c_orderkey)
# from pyspark.sql.functions import sum
@dlt.table(
  table_properties={"quality": "gold"},
  comment="orders aggregated table"
)
def orders_agg_gold():
    df = spark.read.table("LIVE.joined_silver")
    df_final = df.groupBy("c_mktsegment") \
                 .agg(count("o_orderkey").alias("count_orders"),sum("o_totalprice").alias("sum_totalprice")) \
                 .withColumn("__insert_date", current_timestamp())
    return df_final

In [0]:
for _status in _order_status.split(","):
    @dlt.table(
    table_properties = {"quality": "gold"},
    comment = "orders aggregated table",
    name = f"orders_agg_{_status}_gold"
    )
    def func():
        df = spark.read.table("LIVE.joined_silver")

        df_final = df.where(f"o_orderstatus = '{_status}'").groupBy("c_mktsegment").agg(count("o_orderkey").alias("count_orders"),
        sum("o_totalprice").alias("sum_totalprice")).withColumn("__insert_date", current_timestamp())
        
        return df_final
        

In [0]:
# Create a streaming table for Orders
# @dlt.table(
#   table_properties = {"quality": "bronze"},
#   comment = "Order bronze table"
# )
# def orders_bronze():
#   df = spark.readStream.table("dev.bronze.orders_raw")
#   return df


In [0]:
# Create a Materialized View for Customers
# @dlt.table(
#   table_properties = {"quality": "bronze"},
#   comment = "Customer bronze table",
#   name = "customer_bronze"
# )
# def cust_bronze():
#   df = spark.read.table("dev.bronze.customer_raw")
#   return df


In [0]:
# Aggregate based on c_mktsegment and find the count of order (c_orderkey)
# @dlt.table(
#   table_properties = {"quality": "gold"},
#   comment = "orders aggregated table"
# )
# def orders_agg_gold():
#   df = spark.read.table("LIVE.joined_silver")

#   df_final = df.groupBy("c_mktsegment").agg(count("c_orderkey").alias("sum_orders")).withColumn("__insert_date", current_timestamp())

#   return df_final
